# Hanium AML Targeted Face Attack Pipeline

Clean Colab workflow for the attack owner. This notebook restores the repo, LFW data, and trained ResNet-50 checkpoint from personal Google Drive, then runs the targeted attack evaluation pipeline.

Expected Drive inputs:

```text
/content/drive/MyDrive/hanium-aml/archive.zip
/content/drive/MyDrive/hanium-aml/results/checkpoints/face_resnet50_lfw10/best.pt
```

Expected final outputs:

```text
/content/drive/MyDrive/hanium-aml/results/compact_attack_summary.csv
/content/drive/MyDrive/hanium-aml/results/face_attack_summary.csv
/content/drive/MyDrive/hanium-aml/results/attack_index.csv
/content/drive/MyDrive/hanium-aml/results/representative_samples.csv
/content/drive/MyDrive/hanium-aml/results/figures/
/content/drive/MyDrive/hanium-aml/results/attack_panels/representatives/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!nvidia-smi

import torch
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
%cd /content

!if [ ! -d /content/26_HC160/.git ]; then   git clone https://github.com/Yaho03/26_HC160.git /content/26_HC160; else   cd /content/26_HC160 && git pull; fi

%cd /content/26_HC160

## Restore Dataset And Checkpoint

This cell is intentionally Python-based because Colab runtime resets often. It restores the zip, detects the actual LFW raw directory, rebuilds the ImageFolder dataset, and restores the trained checkpoint.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

repo = Path("/content/26_HC160")
drive_zip = Path("/content/drive/MyDrive/hanium-aml/archive.zip")
drive_ckpt = Path("/content/drive/MyDrive/hanium-aml/results/checkpoints/face_resnet50_lfw10")

assert repo.exists(), "repo missing: /content/26_HC160"
assert drive_zip.exists(), f"archive.zip missing: {drive_zip}"
assert drive_ckpt.exists(), f"checkpoint folder missing: {drive_ckpt}"

os.chdir(repo)

print("=== restore archive ===")
raw_root = repo / "data" / "raw_unzipped"
raw_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(drive_zip, "r") as z:
    z.extractall(raw_root)

print("=== restore checkpoint ===")
(repo / "checkpoints").mkdir(exist_ok=True)
dst_ckpt = repo / "checkpoints" / "face_resnet50_lfw10"
if dst_ckpt.exists():
    shutil.rmtree(dst_ckpt)
shutil.copytree(drive_ckpt, dst_ckpt)

print("=== detect LFW raw dir ===")
candidates = []
for p in raw_root.rglob("*"):
    if not p.is_dir():
        continue
    person_dirs = [d for d in p.iterdir() if d.is_dir()]
    valid_people = 0
    jpg_count = 0
    for d in person_dirs[:300]:
        imgs = list(d.glob("*.jpg"))
        if imgs:
            valid_people += 1
            jpg_count += len(imgs)
    if valid_people >= 100:
        candidates.append((valid_people, jpg_count, p))

if not candidates:
    raise RuntimeError("Could not find LFW image folder under archive extraction.")

candidates.sort(reverse=True)
raw_dir = candidates[0][2]
print("Detected LFW raw dir:", raw_dir)

print("=== prepare processed dataset ===")
subprocess.run([
    "python", "-m", "src.datasets.prepare_lfw_identity_dataset",
    "--raw-dir", str(raw_dir),
    "--out-dir", "data/processed/lfw_identity_10",
], check=True)

print("=== final check ===")
print("checkpoint:", (repo / "checkpoints/face_resnet50_lfw10/best.pt").exists())
print("test image count:", len(list((repo / "data/processed/lfw_identity_10/test").rglob("*.jpg"))))

In [ ]:
!cd /content/26_HC160 && python -c "import torch; print('cuda:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!cd /content/26_HC160 && find data/processed/lfw_identity_10/test -type f -name '*.jpg' | wc -l
!cd /content/26_HC160 && ls -lh checkpoints/face_resnet50_lfw10/best.pt

## Run Targeted Attacks

The LFW-10 test split has 223 images. `--limit 300` therefore evaluates the full test set.

In [ ]:
!cd /content/26_HC160 && PYTHONUNBUFFERED=1 python -u -m src.attacks.targeted_fgsm_face --epsilon 0.005 --limit 300

In [ ]:
!cd /content/26_HC160 && PYTHONUNBUFFERED=1 python -u -m src.attacks.targeted_pgd_face --epsilon 0.03 --alpha 0.003 --steps 10 --limit 300

In [ ]:
!cd /content/26_HC160 && PYTHONUNBUFFERED=1 python -u -m src.attacks.targeted_square_face --epsilon 0.05 --max-queries 300 --limit 300

In [ ]:
!cd /content/26_HC160 && PYTHONUNBUFFERED=1 python -u -m src.attacks.targeted_jsma_face --theta 0.05 --steps 20 --pixels-per-step 200 --limit 300

In [ ]:
!cd /content/26_HC160 && PYTHONUNBUFFERED=1 python -u -m src.attacks.targeted_zoo_face --epsilon 0.05 --max-queries 2000 --coords-per-iter 128 --learning-rate 0.02 --limit 300

## Summarize, Build Handoff Index, And Save To Drive

In [ ]:
!cd /content/26_HC160 && python -m src.reports.summarize_face_attack
!cd /content/26_HC160 && python -m src.reports.make_compact_attack_summary
!cd /content/26_HC160 && python -m src.reports.build_attack_index
!cd /content/26_HC160 && python -m src.reports.select_representative_attacks
!cd /content/26_HC160 && python -m src.reports.make_representative_panels
!cd /content/26_HC160 && python -m src.reports.plot_attack_summary

!mkdir -p /content/drive/MyDrive/hanium-aml/results/figures
!mkdir -p /content/drive/MyDrive/hanium-aml/results/attack_panels

!cp /content/26_HC160/outputs/attacks/face_attack_summary.csv /content/drive/MyDrive/hanium-aml/results/
!cp /content/26_HC160/outputs/attacks/compact_attack_summary.csv /content/drive/MyDrive/hanium-aml/results/
!cp /content/26_HC160/outputs/attacks/attack_index.csv /content/drive/MyDrive/hanium-aml/results/
!cp /content/26_HC160/outputs/attacks/representative_samples.csv /content/drive/MyDrive/hanium-aml/results/
!cp -r /content/26_HC160/outputs/attack_panels/representatives /content/drive/MyDrive/hanium-aml/results/attack_panels/
!cp -r /content/26_HC160/outputs/attacks/figures/* /content/drive/MyDrive/hanium-aml/results/figures/

!echo "=== compact summary ==="
!cat /content/26_HC160/outputs/attacks/compact_attack_summary.csv

!echo "=== index count ==="
!python -c "import pandas as pd; df=pd.read_csv('/content/26_HC160/outputs/attacks/attack_index.csv'); print(df.groupby('attack_family').size()); print('total:', len(df))"